# 2.1 Data Manipulation

Before anything else can happen — training a model, computing a gradient, evaluating a prediction — we need a way to hold numbers in memory and operate on them efficiently. PyTorch's answer is the **tensor** ($n$-dimensional array): conceptually the same as NumPy's `ndarray`, but with two capabilities NumPy lacks — native **GPU** acceleration and built-in **automatic differentiation**. These properties make neural networks both easy to code and fast to run, and the tensor class is the main interface this entire book uses for storing and manipulating data.

This notebook works through the basic tensor mechanics: **creating** tensors, **indexing** and **slicing** into them, elementwise **operations**, the **broadcasting** mechanism that lets differently-shaped tensors interoperate, **memory-efficient** in-place updates, and **conversion** to and from NumPy arrays and native Python scalars.

In [1]:
import torch
import numpy as np

## 2.1.1 Getting Started

A tensor represents a (possibly multidimensional) array of numerical values. In the one-dimensional case, with only one axis, a tensor is called a **vector**; with two axes, a **matrix**; with $k > 2$ axes, we drop the specialized names and just call it a $k$th-order tensor. PyTorch provides a variety of functions for creating new tensors prepopulated with values, similar to NumPy's `arange`, `zeros`, `ones`, and random-sampling functions. Unless otherwise specified, new tensors are stored in main memory and designated for CPU-based computation.

In [2]:
# create a 1-D tensor with 32-bit floating-point numbers
x = torch.arange(12, dtype=torch.float32)
x

tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11.])

In [3]:
# NUMber of ELements in tensor
x.numel()

12

In [4]:
# shape of tensor
x.shape

torch.Size([12])

In [5]:
# Reshape the tensor to 3 rows and 4 columns
X = x.reshape(3, 4)
X

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])

Specifying every shape component to `reshape` is redundant once the tensor's total size is fixed: given the size and all but one of the target dimensions, the last one is determined. Passing `-1` for that dimension tells PyTorch to infer it automatically, so `x.reshape(3, 4)` above is equivalent to `x.reshape(-1, 4)` or `x.reshape(3, -1)`.

In [6]:
x.reshape(-1, 4), x.reshape(3, -1)   # -1 tells PyTorch to infer that axis from numel() and the other dimension

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]),
 tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

In [7]:
# Create a 3-D tensor of shape (2, 3, 4) filled with zeros
torch.zeros((2, 3, 4))

tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [8]:
# Same shape (2, 3, 4), filled with ones instead of zeros
torch.ones((2, 3, 4))

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [1., 1., 1., 1.]]])

In [9]:
# Create a 2-D tensor with 3 rows and 4 columns filled with random numbers 
# from a standard normal distribution
torch.randn(3, 4)

tensor([[ 0.3558,  1.6117, -0.0354,  0.7508],
        [-0.4660,  0.8342, -1.5829, -0.8760],
        [-0.2976, -0.2456, -0.5973, -0.3332]])

Tensors can also be built directly from Python data: supplying (possibly nested) list(s) of numerical literals specifies every element by hand. In a list of lists, the outer list corresponds to axis 0 (rows) and each inner list to axis 1 (columns).

In [10]:
torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])   # nested list -> outer list = axis 0 (rows), inner lists = axis 1 (columns)

tensor([[2, 1, 4, 3],
        [1, 2, 3, 4],
        [4, 3, 2, 1]])

## 2.1.2 Indexing and Slicing

As with Python lists, tensor elements can be accessed by indexing (starting at 0), and negative indices count backward from the end. Slices work the same way too: `[start:stop]` selects elements from `start` up to but *not including* `stop`. When only one index (or slice) is given for a $k$th-order tensor, it applies along axis 0 — so `X[-1]` selects the last row and `X[1:3]` the second and third rows. Beyond reading, indexing can also be used to *write* into a tensor — assigning to a single element, or to a whole slice at once, mutates it in place.

In [11]:
# last row and slice of 2nd and 3rd columns
X[-1], X[1:3]

(tensor([ 8.,  9., 10., 11.]),
 tensor([[ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.]]))

In [12]:
X[1, 2] = 17   # write a single element: row 1, column 2
X

tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5., 17.,  7.],
        [ 8.,  9., 10., 11.]])

In [13]:
# set first two rows to 12
X[:2, :] = 12
X

tensor([[12., 12., 12., 12.],
        [12., 12., 12., 12.],
        [ 8.,  9., 10., 11.]])

## 2.1.3 Operations

Tensors support the usual math. **Unary** elementwise operations (signature $f: \mathbb{R} \to \mathbb{R}$, like `exp`) map every element through the same scalar function. **Binary** elementwise operations (signature $f: \mathbb{R}, \mathbb{R} \to \mathbb{R}$ — the standard `+`, `-`, `*`, `/`, `**`) take two tensors of the *same shape* and combine them position by position, producing an output of that same shape. (Linear-algebraic operations like dot products and matrix multiplication are a different story, covered in the linear algebra section that follows this one.) Beyond elementwise arithmetic, tensors can also be **concatenated** end to end along a chosen axis, compared elementwise with logical operators, and **summed** over all elements to collapse down to a single scalar.

In [14]:
# Compute the exponential of each element in the tensor
torch.exp(x)

tensor([162754.7969, 162754.7969, 162754.7969, 162754.7969, 162754.7969,
        162754.7969, 162754.7969, 162754.7969,   2980.9580,   8103.0840,
         22026.4648,  59874.1406])

In [15]:
# Element-wise operations
x = torch.tensor([1.0, 2, 4, 8])
y = torch.tensor([2, 2, 2, 2])
x + y, x- y, x * y, x / y, x ** y

(tensor([ 3.,  4.,  6., 10.]),
 tensor([-1.,  0.,  2.,  6.]),
 tensor([ 2.,  4.,  8., 16.]),
 tensor([0.5000, 1.0000, 2.0000, 4.0000]),
 tensor([ 1.,  4., 16., 64.]))

In [16]:
X = torch.arange(12, dtype=torch.float32).reshape((3,4))
Y = torch.tensor([[2.0, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])
# concatenate two tensors along rows (dim=0) and along columns (dim=1)
torch.cat((X, Y), dim=0), torch.cat((X, Y), dim=1)

(tensor([[ 0.,  1.,  2.,  3.],
         [ 4.,  5.,  6.,  7.],
         [ 8.,  9., 10., 11.],
         [ 2.,  1.,  4.,  3.],
         [ 1.,  2.,  3.,  4.],
         [ 4.,  3.,  2.,  1.]]),
 tensor([[ 0.,  1.,  2.,  3.,  2.,  1.,  4.,  3.],
         [ 4.,  5.,  6.,  7.,  1.,  2.,  3.,  4.],
         [ 8.,  9., 10., 11.,  4.,  3.,  2.,  1.]]))

In [17]:
# Compare two tensors element-wise
X == Y

tensor([[False,  True, False,  True],
        [False, False, False, False],
        [False, False, False, False]])

In [18]:
X.sum()   # sum over every element -> a single 0-D (scalar) tensor

tensor(66.)

## 2.1.4 Broadcasting

Elementwise binary operations normally require both tensors to have the exact same shape. Under certain conditions, **broadcasting** lets us perform elementwise operations even when shapes differ: (i) expand one or both arrays by copying elements along axes of length 1, so that the two tensors end up with the same shape, then (ii) apply the elementwise operation on the results. Below, a $3\times 1$ column vector and a $1\times 2$ row vector broadcast to a shared $3\times 2$ shape by replicating the column vector across columns and the row vector across rows.

In [19]:
# Create a tensor of shape (3, 1) -> This is a COLUMN vector
# Values: [[0], [1], [2]]
a = torch.arange(3).reshape((3, 1))

# Create a tensor of shape (1, 2) -> This is a ROW vector
# Values: [[0, 1]]
b = torch.arange(2).reshape((1, 2))

# --- The Broadcasting Operation ---
# Since shapes (3,1) and (1,2) are different but compatible, 
# PyTorch expands them both to shape (3,2) automatically.
# 'a' is copied across columns, 'b' is copied across rows.
result = a + b 

# Final Shape: (3, 2)
# Final Values:
# [[0+0, 0+1],  -> [[0, 1],
#  [1+0, 1+1],  ->  [1, 2],
#  [2+0, 2+1]]  ->  [2, 3]]
result


tensor([[0, 1],
        [1, 2],
        [2, 3]])

## 2.1.5 Saving Memory

Running operations can allocate new memory for the results: `Y = Y + X` computes `X + Y` into a freshly allocated tensor and only *then* rebinds the name `Y` to point at it — the old `Y` is dereferenced and, once nothing else references it, garbage collected. That's wasteful for two reasons. First, in machine learning we often hold hundreds of megabytes of parameters and update all of them multiple times per second, so we would rather not allocate memory needlessly on every update. Second, if other variables still hold references to the old memory, failing to update in place risks some of them silently pointing at stale parameters. PyTorch lets us update tensors **in place** instead, either by assigning into a slice (`Z[:] = ...`) or through in-place operators such as `+=` — both keep the tensor's memory address unchanged.

In [20]:
# Y gets a new memory address after the operation
before = id(Y)
Y = Y + X
print(id(Y) == before)
print(id(Y))

False
129345991716080


In [21]:
# In-place operation: Y keeps the same memory address
Z = torch.zeros_like(Y) # create a tensor of the same shape with 0.
print('id(Z):', id(Z))
Z[:] = X + Y
print('id(Z):', id(Z))

id(Z): 129345991719840
id(Z): 129345991719840


In [22]:
# In-place operation: X keeps the same memory address if use +=
before = id(X)
X += Y
id(X) == before

True

## 2.1.6 Conversion to Other Python Objects

Converting to a NumPy array, or vice versa, is easy: the torch tensor and the resulting NumPy array **share their underlying memory**, so changing one through an in-place operation also changes the other.

In [23]:
A = X.numpy()             # tensor -> NumPy array (shares memory with X on CPU)
B = torch.from_numpy(A)   # NumPy array -> tensor (shares memory with A)
type(A), type(B)

(numpy.ndarray, torch.Tensor)

To convert a size-1 tensor to a Python scalar, invoke the `item` function or Python's built-in `float`/`int`.

In [24]:
a = torch.tensor([3.5])
a, a.item(), float(a), int(a)   # size-1 tensor and its Python-native equivalents

(tensor([3.5000]), 3.5, 3.5, 3)

## 2.1.7 Summary

- The **tensor** class is the main interface for storing and manipulating data: an $n$-dimensional array like NumPy's `ndarray`, plus GPU acceleration and automatic differentiation.
- **Construction routines**: `arange`, `zeros`, `ones`, `randn`, and nested Python lists all build tensors; `reshape` (with an optional inferred `-1` axis) changes shape without changing the underlying data or size.
- **Indexing and slicing** work like Python lists — zero-based, negative indices count from the end, `[start:stop]` excludes `stop` — and can be used to write into a tensor as well as read from it.
- **Basic math operations**: unary elementwise ops like `exp`, binary elementwise ops (`+`/`-`/`*`/`/`/`**`) on same-shaped tensors, `cat` to join tensors along an axis, elementwise comparisons, and `sum` to reduce to a scalar.
- **Broadcasting** lets differently-shaped tensors interoperate by expanding size-1 axes to match before applying the operation.
- **Memory-efficient assignment**: `Y = Y + X` allocates new memory, while slice assignment (`Z[:] = ...`) or in-place operators (`+=`) update a tensor in place instead.
- **Conversion to and from other Python objects**: `.numpy()` / `torch.from_numpy()` convert to/from NumPy arrays (sharing memory on the CPU), and `.item()`, `float()`, or `int()` convert a size-1 tensor to a native Python scalar.

## 2.1.8 Exercises
- 1. Run the code in this section. Change the conditional statement X == Y to X < Y or X > Y, and then see what kind of tensor you can get.
- 2. Replace the two tensors that operate by element in the broadcasting mechanism with
other shapes, e.g., 3-dimensional tensors. Is the result the same as expected?

In [25]:
# 1: 2D tensor comparison
X = torch.arange(12, dtype=torch.float32).reshape((3, 4))
Y = torch.tensor([[2, 1, 4, 3], [1, 2, 3, 4], [4, 3, 2, 1]])

print(X < Y)
print(X > Y)

tensor([[ True, False,  True, False],
        [False, False, False, False],
        [False, False, False, False]])
tensor([[False, False, False, False],
        [ True,  True,  True,  True],
        [ True,  True,  True,  True]])


In [26]:
# 2: Broadcasting example
# A is (3, 1, 4)
A = torch.arange(12).reshape((3, 1, 4))

# B is (1, 2, 1)
B = torch.arange(2).reshape((1, 2, 1))

C = A + B
print(f"Shape of A: {A.shape}")
print(f"Shape of B: {B.shape}")
print(f"Shape of C: {C.shape}")
print(C)

Shape of A: torch.Size([3, 1, 4])
Shape of B: torch.Size([1, 2, 1])
Shape of C: torch.Size([3, 2, 4])
tensor([[[ 0,  1,  2,  3],
         [ 1,  2,  3,  4]],

        [[ 4,  5,  6,  7],
         [ 5,  6,  7,  8]],

        [[ 8,  9, 10, 11],
         [ 9, 10, 11, 12]]])
